# CellIQ tissue model: repeat the test and check the heatmaps

Two questions, answered with measurements:

1. **How stable is the accuracy?** Train the model several times, each time testing on a **different group of images** (the near-duplicate-aware split has 7 groups), and report the range instead of one number. Two recipes are repeated: ours (the one in the app) and the "v2 grayscale-robust" one.
2. **Do the heatmaps point at what the model uses?** Hide the red part of the heatmap and see whether the answer breaks, compared with hiding a different area of the same size.

**How to run:** Runtime > Change runtime type > T4 GPU, then Runtime > Run all. About 45 minutes.

Research and educational prototype. Not for real diagnosis.

## 1. Settings

In [ ]:
DATASET = "lung_slides"
IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42
TEST_FOLDS = [0, 1, 2]          # which of the 7 near-duplicate-aware groups serve as the test set, one run each
OUT_DIR = "/content/robust_out"
HEATMAP_N = 600                 # test images used for the heatmap check
MASK_FRACTION = 0.20            # share of each image hidden in the heatmap check

## 2. Tools, data and file list (same code as our tissue notebook)

In [ ]:
import os, json, random, re, shutil
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)
os.makedirs(OUT_DIR, exist_ok=True)
from sklearn.model_selection import train_test_split
from concurrent.futures import ThreadPoolExecutor

In [ ]:
import kagglehub
path = kagglehub.dataset_download("javaidahmadwani/lc25000")
DATA_DIR = None
for root, dirs, _ in os.walk(path):
    if {"lung_aca", "lung_n", "lung_scc"} <= set(dirs):
        DATA_DIR = root
        break
print("DATA_DIR =", DATA_DIR)
assert DATA_DIR, "lung folders not found"

In [ ]:
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")
class_names = sorted([d for d in os.listdir(DATA_DIR)
                      if os.path.isdir(os.path.join(DATA_DIR, d)) and d.startswith("lung_")])
print("Groups:", class_names)
natural = lambda s: [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", s)]
files, labels = [], []
for idx, d in enumerate(class_names):
    folder = os.path.join(DATA_DIR, d)
    for f in sorted(os.listdir(folder), key=natural):
        if f.lower().endswith(IMG_EXT):
            files.append(os.path.join(folder, f)); labels.append(idx)
files = np.array(files); labels = np.array(labels)
print("Total images:", len(files), "| per group:", np.bincount(labels))

## 3. Load every image once at 224 x 224 (faster training, identical resize)

In [ ]:
def load_resized(path):
    return np.asarray(Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR))

with ThreadPoolExecutor(max_workers=8) as ex:
    CACHE = np.stack(list(ex.map(load_resized, files)))
print("cached images:", CACHE.shape)

## 4. Group near-duplicates and make the 7 groups (our method, unchanged)

In [ ]:
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
emb_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize(mean, std)])

class PlainDS(Dataset):
    def __len__(self): return len(files)
    def __getitem__(self, i): return emb_tf(Image.fromarray(CACHE[i])), 0

emb = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT).to(device).eval()
emb.classifier = nn.Identity()
feats = []
with torch.no_grad():
    for x, _ in DataLoader(PlainDS(), batch_size=64, num_workers=2):
        feats.append(emb(x.to(device)).float().cpu())
feats = torch.cat(feats)
feats = F.normalize(feats - feats.mean(0), dim=1).to(device)
N = len(feats)

FLOOR = 0.75
pi, pj, ps = [], [], []
for i in range(0, N, 1024):
    s = feats[i:i+1024] @ feats.T
    r, c = torch.where(s >= FLOOR)
    pi.append((r + i).cpu().numpy()); pj.append(c.cpu().numpy()); ps.append(s[r, c].cpu().numpy())
pi, pj, ps = np.concatenate(pi), np.concatenate(pj), np.concatenate(ps)
print("Linked pairs at similarity >=", FLOOR, ":", len(pi))

def components(thr):
    m = ps >= thr
    G = coo_matrix((np.ones(m.sum()), (pi[m], pj[m])), shape=(N, N))
    n, gid = connected_components(G, directed=False)
    return n, gid

table = {}
for thr in (0.80, 0.825, 0.85, 0.875, 0.90, 0.925, 0.95):
    n, gid = components(thr)
    sizes = np.bincount(gid)
    table[thr] = (n, gid, int(sizes.max()))
    print(f"similarity >= {thr:.3f}: groups={n:5d} | biggest group={sizes.max():5d} | median size={int(np.median(sizes))}")

# Linking images by similarity can chain unrelated photos into one giant group (which breaks the split).
# Use the LOWEST threshold (most careful grouping) whose biggest group is still small enough to split fairly.
MAX_GROUP = 700
ok_thr = [t for t in table if table[t][2] <= MAX_GROUP]
SIM = min(ok_thr) if ok_thr else max(table)
n_groups, group_id, _ = table[SIM]
print("Chosen similarity threshold:", SIM, "->", n_groups, "groups")

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
folds = [idx for _, idx in sgkf.split(files, labels, group_id)]
te, va = folds[0], folds[1]
tr = np.concatenate(folds[2:])
for nm, ix in (("train", tr), ("val", va), ("test", te)):
    print(f"{nm:5s} n={len(ix):5d} per group: {np.bincount(labels[ix], minlength=len(class_names))}")

# leakage check: how many test images still have a similar-looking twin in train? (random split vs group split)
def twin_rates(test_idx, train_idx, thrs=(0.95, 0.90, 0.85, 0.80)):
    best = []
    for i in range(0, len(test_idx), 512):
        s = feats[test_idx[i:i+512]] @ feats[train_idx].T
        best.append(s.max(1).values.cpu().numpy())
    best = np.concatenate(best)
    return {str(t): round(float((best >= t).mean()), 4) for t in thrs}

rng = np.random.RandomState(SEED); perm = rng.permutation(N)
r_te, r_tr = perm[:len(te)], perm[len(te) + len(va):]
leak_random = twin_rates(r_te, r_tr)
leak_group = twin_rates(te, tr)
print("Share of test images with a near-identical twin in train, at similarity >= 0.95 / 0.90 / 0.85 / 0.80:")
print("  random split:", leak_random)
print("  group split :", leak_group, "(zero at or above the grouping threshold by construction; lower levels show what remains)")

## 5. The two recipes, training and checks

In [ ]:
eval_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize(mean, std)])
gray_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.Grayscale(3), transforms.ToTensor(), transforms.Normalize(mean, std)])
K = len(class_names)

RECIPES = {
    "ours (app model)": dict(
        tf=transforms.Compose([transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)), transforms.RandomHorizontalFlip(),
                               transforms.RandomVerticalFlip(), transforms.RandomRotation(20), transforms.RandomGrayscale(p=0.3),
                               transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.5, hue=0.1),
                               transforms.ToTensor(), transforms.Normalize(mean, std)]),
        head=3, finetune=4, smooth=0.0, cosine=False),
    "v2 grayscale-robust": dict(
        tf=transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.RandomGrayscale(p=0.5), transforms.RandomHorizontalFlip(),
                               transforms.RandomRotation(10), transforms.ColorJitter(brightness=0.25, contrast=0.25),
                               transforms.ToTensor(), transforms.Normalize(mean, std)]),
        head=4, finetune=8, smooth=0.1, cosine=True),
}


class CachedDS(Dataset):
    def __init__(self, idxs, tf):
        self.idxs, self.tf = idxs, tf
    def __len__(self):
        return len(self.idxs)
    def __getitem__(self, i):
        j = self.idxs[i]
        return self.tf(Image.fromarray(CACHE[j])), int(labels[j])


def train_recipe(rec, tr_idx, va_idx, seed, tag):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    mdl = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    mdl.classifier[1] = nn.Linear(mdl.classifier[1].in_features, K)
    mdl = mdl.to(device)
    tdl = DataLoader(CachedDS(tr_idx, rec["tf"]), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    vdl = DataLoader(CachedDS(va_idx, eval_tf), batch_size=BATCH_SIZE, num_workers=2)
    cnt = np.bincount(labels[tr_idx], minlength=K)
    w = torch.tensor(cnt.sum() / (K * cnt), dtype=torch.float32).to(device)
    loss_fn = nn.CrossEntropyLoss(weight=w, label_smoothing=rec["smooth"])
    best = {"loss": 1e9, "state": None}

    def run_epoch(dl, train, opt=None):
        mdl.train(train)
        total, correct, loss_sum = 0, 0, 0.0
        with torch.set_grad_enabled(train):
            for x, y in dl:
                x, y = x.to(device), y.to(device)
                out = mdl(x)
                loss = loss_fn(out, y)
                if train:
                    opt.zero_grad(); loss.backward(); opt.step()
                loss_sum += loss.item() * len(y); total += len(y)
                correct += (out.argmax(1) == y).sum().item()
        return loss_sum / total, correct / total

    def fit(epochs, lr, name):
        opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, mdl.parameters()), lr=lr)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs) if rec["cosine"] else None
        for ep in range(epochs):
            tl, ta = run_epoch(tdl, True, opt)
            vl, va_ = run_epoch(vdl, False)
            if sched:
                sched.step()
            if vl < best["loss"]:
                best["loss"] = vl
                best["state"] = {k: v.detach().cpu().clone() for k, v in mdl.state_dict().items()}
        print(f"[{tag}] {name} done: last train acc {ta:.3f} | val acc {va_:.3f}", flush=True)

    for p in mdl.parameters():
        p.requires_grad = False
    for p in mdl.classifier.parameters():
        p.requires_grad = True
    fit(rec["head"], 1e-3, "head")
    for p in mdl.parameters():
        p.requires_grad = True
    fit(rec["finetune"], 1e-4, "fine-tune")
    mdl.load_state_dict(best["state"])
    return mdl


@torch.no_grad()
def get_probs(mdl, idx, tf):
    mdl.eval()
    P, Y = [], []
    for x, y in DataLoader(CachedDS(idx, tf), batch_size=64, num_workers=2):
        P.append(F.softmax(mdl(x.to(device)), dim=1).cpu().numpy())
        Y.append(y.numpy())
    return np.concatenate(P), np.concatenate(Y)


def evaluate(mdl, te_idx, va_idx):
    pt, yt = get_probs(mdl, te_idx, eval_tf)
    pg, _ = get_probs(mdl, te_idx, gray_tf)
    pv, yv = get_probs(mdl, va_idx, eval_tf)
    pred, pred_g = pt.argmax(1), pg.argmax(1)
    n_idx = class_names.index("lung_n")
    risk_v, risk_t = 1 - pv[:, n_idx], 1 - pt[:, n_idx]
    yb_v, yb_t = (yv != n_idx).astype(int), (yt != n_idx).astype(int)
    ok = [t for t in np.linspace(0.01, 0.99, 99) if (risk_v[yb_v == 1] >= t).mean() >= 0.95]
    thr = float(max(ok)) if ok else 0.5
    tn, fp, fn, tp = confusion_matrix(yb_t, (risk_t >= thr).astype(int), labels=[0, 1]).ravel()
    return {"n_test": int(len(te_idx)), "accuracy": round(float((pred == yt).mean()), 4),
            "accuracy_grayscale": round(float((pred_g == yt).mean()), 4), "wrong": int((pred != yt).sum()),
            "sensitivity": round(float(tp / max(tp + fn, 1)), 3), "specificity": round(float(tn / max(tn + fp, 1)), 3),
            "auc": round(float(roc_auc_score(yb_t, risk_t)), 3)}
print("ready")

## 6. Repeat: each recipe, each test group

For test group *k*, the next group is used for validation and the other five for training. Different group = different test images and a different training set.

In [ ]:
runs = []
keep = {}
for k in TEST_FOLDS:
    te_k, va_k = folds[k], folds[(k + 1) % 7]
    tr_k = np.concatenate([folds[j] for j in range(7) if j not in (k, (k + 1) % 7)])
    for name, rec in RECIPES.items():
        mdl = train_recipe(rec, tr_k, va_k, SEED + k, f"{name} | test group {k}")
        res = evaluate(mdl, te_k, va_k)
        res.update({"recipe": name, "test_group": k})
        runs.append(res)
        print("RESULT", {kk: vv for kk, vv in res.items()}, flush=True)
        if name.startswith("ours") and k == TEST_FOLDS[0]:
            keep = {"model": mdl, "test_idx": te_k}      # used for the heatmap check below
        else:
            del mdl
        torch.cuda.empty_cache()

## 7. Summary: the range over the repeats

In [ ]:
import pandas as pd
df = pd.DataFrame(runs)
cols = ["accuracy", "accuracy_grayscale", "sensitivity", "specificity", "auc", "wrong"]
print(df[["recipe", "test_group", "n_test"] + cols].to_string(index=False))
summary = {}
for name, g in df.groupby("recipe"):
    summary[name] = {c: {"mean": round(float(g[c].mean()), 4), "min": round(float(g[c].min()), 4), "max": round(float(g[c].max()), 4)} for c in cols}
    print("\n" + name)
    for c in cols:
        print(f"   {c:20s} mean {g[c].mean():.4f} | range {g[c].min():.4f} to {g[c].max():.4f}")

## 8. Heatmap check

For the app's recipe (first test group) we compute the Grad-CAM heatmap for each test image, then:

- **Border share:** how much of the heatmap sits in the outer frame of the image. A uniform heatmap would put about 49% there; much more than that means it is looking at edges.
- **Hide test:** hide the 20% of pixels the heatmap likes best and see how many correct answers break. As controls, hide the same amount in the *least* important area, and a heatmap borrowed from a *different* image (same blob shape, wrong place).
  If the heatmap is meaningful, hiding its top area should break far more answers than the controls.

In [ ]:
mdl, te_idx = keep["model"], keep["test_idx"]
sub = np.random.RandomState(SEED).choice(te_idx, size=min(HEATMAP_N, len(te_idx)), replace=False)
mdl.eval()
store = {}
def save_maps(module, inputs, output):          # must return nothing, or it would replace the layer output
    store["a"] = output
    output.register_hook(lambda grad: store.__setitem__("g", grad))
hook = mdl.features[-1].register_forward_hook(save_maps)

cams, xs, ys, preds, probs0 = [], [], [], [], []
for x, y in DataLoader(CachedDS(sub, eval_tf), batch_size=50, num_workers=2):
    x = x.to(device)
    mdl.zero_grad()
    out = mdl(x)
    pred = out.argmax(1)
    out.gather(1, pred[:, None]).sum().backward()
    w = store["g"].mean(dim=(2, 3), keepdim=True)
    cam = F.relu((w * store["a"]).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)[:, 0].detach()
    mn = cam.flatten(1).min(1).values[:, None, None]; mx = cam.flatten(1).max(1).values[:, None, None]
    cams.append(((cam - mn) / (mx - mn + 1e-8)).cpu()); xs.append(x.detach().cpu()); ys.append(y)
    preds.append(pred.cpu()); probs0.append(F.softmax(out, 1).detach().gather(1, pred[:, None])[:, 0].cpu())
hook.remove()
cams, xs, ys, preds, probs0 = torch.cat(cams), torch.cat(xs), torch.cat(ys), torch.cat(preds), torch.cat(probs0)
print("heatmaps computed for", len(cams), "test images")

# 1) border share
m = 32
border = torch.ones(IMG_SIZE, IMG_SIZE); border[m:-m, m:-m] = 0
share = ((cams * border).flatten(1).sum(1) / (cams.flatten(1).sum(1) + 1e-8)).mean().item()
print(f"share of heatmap in the outer {m}px frame: {share:.2f} (a uniform heatmap would give {border.mean().item():.2f})")

# 2) hide test
@torch.no_grad()
def predict_masked(mask):
    out_pred, out_prob = [], []
    for i in range(0, len(xs), 100):
        xb = (xs[i:i + 100] * (1 - mask[i:i + 100, None].float())).to(device)      # 0 = mean colour after normalisation
        p = F.softmax(mdl(xb), 1).cpu()
        out_pred.append(p.argmax(1)); out_prob.append(p)
    return torch.cat(out_pred), torch.cat(out_prob)

def top_mask(c, frac):
    thr = torch.quantile(c.flatten(1), 1 - frac, dim=1)[:, None, None]
    return c >= thr

correct = (preds == ys)
results = {}
for name, mask in (("hide top-heatmap area", top_mask(cams, MASK_FRACTION)),
                   ("hide least-important area", top_mask(1 - cams, MASK_FRACTION)),
                   ("hide area from another image's heatmap", top_mask(cams.roll(1, dims=0), MASK_FRACTION))):
    p2, pr2 = predict_masked(mask)
    broke = float(((p2 != ys) & correct).sum() / correct.sum())
    conf = float((probs0[correct] - pr2[correct].gather(1, preds[correct][:, None])[:, 0]).mean())
    results[name] = {"correct_answers_that_break": round(broke, 4), "average_confidence_drop": round(conf, 4)}
    print(f"{name:42s}: {broke * 100:5.1f}% of correct answers break | confidence drops by {conf:.3f}")
heat = {"border_share": round(share, 3), "uniform_border_share": round(border.mean().item(), 3), "hide_test": results, "n_images": int(len(cams)),
        "mask_fraction": MASK_FRACTION}

## 9. Save

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
out = {"runs": runs, "summary": summary, "heatmap_check": heat,
       "note": "Each run trains on 5 of the 7 near-duplicate-aware groups, validates on 1 and tests on 1. Test groups overlap in nothing. One public dataset, not clinically validated."}
json.dump(out, open(f"{OUT_DIR}/robust_results.json", "w"), indent=1)
shutil.make_archive("/content/cellIQ_tissue_robust", "zip", OUT_DIR)
print("saved; zip:", os.path.getsize("/content/cellIQ_tissue_robust.zip"), "bytes")